In [ ]:
#@title Dataset Download & Separation - 550 Questions for db1 and db2, 500 for db3
import gc
import json
import torch
from datasets import load_dataset, load_from_disk

def download_and_separate_dataset():
  full_dataset = load_dataset("ytu-ce-cosmos/gsm8k_tr", split="train")
  full_dataset = full_dataset.shuffle(seed=59)

  dataset_1 = full_dataset.select(range(550))
  dataset_2 = full_dataset.select(range(550, 1100))
  dataset_3 = full_dataset.select(range(1100, 1600))

  dataset_1.save_to_disk("/content/data/db1_distill")
  dataset_2.save_to_disk("/content/data/db2_grpo")
  dataset_3.save_to_disk("/content/data/db3_test")

  print(f"Dataset downloading and separation is completed. Dataset sizes are as follows: DB1: {len(dataset_1)}, DB2: {len(dataset_2)}, DB3:{len(dataset_3)}.")

download_and_separate_dataset()

In [ ]:
#@title Teacher Inference for DB1 & DB2 - Ran Twice
import os
import json
import gc
import torch
from datasets import load_dataset, load_from_disk
from openai import OpenAI
from google.colab import userdata
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

def teacher_inference_for_distill(item):
  question = item['question']
  ground_truth = item['answer']

  sft_record = None
  try:
    sft_response = openrouter_client.chat.completions.create(
    model=teacher_model,
    messages=[
      {"role": "system", "content": system_prompt},
      {"role": "user", "content": question}
    ],
    temperature=0.2,
    max_tokens=1280
  )
  sft_record = {
    "question": question,
    "ground_truth": ground_truth,
    "teacher_answer": sft_response.choices[0].message.content
  }
  except Exception as ex:
    print(f"\nSFT Error on '{question[:20]}...': {ex}")

  return sft_record

os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
openrouter_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ.get("OPENROUTER_API_KEY"))
teacher_model = "openai/gpt-oss-120b"

print("Loading dataset...")
try:
  # teacher_dataset = load_from_disk("/content/data/db1_distill")
  teacher_dataset = load_from_disk("/content/data/db2_grpo")
except Exception:
  # teacher_dataset = load_dataset("ytu-ce-cosmos/gsm8k_tr", split="train").shuffle(seed=59).select(range(550))
  teacher_dataset = load_dataset("ytu-ce-cosmos/gsm8k_tr", split="train").shuffle(seed=59).select(range(550, 1100))

# teacher_output_file = "teacher_distill_answers.jsonl"
teacher_output_file = "teacher_grpo_answers.jsonl"

system_prompt = (
  "You are an expert mathematical assistant. You must solve the problem step-by-step and be extremely concise, in English. "
  "Show your clear chain-of-thought reasoning. "
  "At the very end of your response, you MUST provide the final numerical answer inside explicit XML tags like this: <answer>YOUR_NUMERICAL_ANSWER</answer>. "
  "Do not include units or words inside the tag, only the absolute final number."
)

print(f"Starting teacher answer generation for {len(teacher_dataset)} questions...")

with open(teacher_output_file, 'a', encoding='utf-8') as sft_file:
  with ThreadPoolExecutor(max_workers=15) as executor:
    futures = {executor.submit(teacher_inference_for_distill, item): item for item in teacher_dataset}
    for future in tqdm(as_completed(futures), total=len(teacher_dataset), desc="Processing"):
      sft_result = future.result()
      if sft_result:
        sft_file.write(json.dumps(sft_result, ensure_ascii=False) + '\n')

print("Data generation completed, and the answer files saved in .jsonl format.")

if 'teacher_model' in locals():
  del teacher_model
  gc.collect()
  torch.cuda.empty_cache()

In [ ]:
#@title Filtering the Dataset to 500 Questions with Answers - LLM GENERATED
import json

def clean_and_trim_dataset(input_filename, output_filename, exact_count=500):
  perfect_samples = []

  with open(input_filename, 'r', encoding='utf-8') as f:
    for line in f:
      data = json.loads(line)
      teacher_answer = data.get('teacher_answer')

      if teacher_answer and "</answer>" in teacher_answer:
        perfect_samples.append(data)

      if len(perfect_samples) == exact_count:
        break

  with open(output_filename, 'w', encoding='utf-8') as f:
    for item in perfect_samples:
      f.write(json.dumps(item, ensure_ascii=False) + '\n')

  print(f"Filtered and saved exactly {len(perfect_samples)} perfect samples to {output_filename}.")

clean_and_trim_dataset("teacher_grpo_answers.jsonl", "db2_distill_cleaned_dataset.jsonl")

In [ ]:
#@title Distillation Process for Base Model - Perform SFT with db1
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

def apply_chat_template(example):
  messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": example['question']},
    {"role": "assistant", "content": example['teacher_answer']}
  ]
  example["text"] = sft_tokenizer.apply_chat_template(messages, tokenize=False)
  return example

print("Loading dataset...")
db1_distill_dataset = load_dataset("json", data_files="db1_distill_cleaned_dataset.jsonl", split="train")

base_model_name = "Qwen/Qwen3.5-4B"
print(f"Loading {base_model_name}...")

sft_tokenizer = AutoTokenizer.from_pretrained(base_model_name)
sft_tokenizer.pad_token = sft_tokenizer.eos_token
sft_model = AutoModelForCausalLM.from_pretrained(base_model_name, torch_dtype=torch.bfloat16, device_map="auto")

system_prompt = (
  "You are an expert mathematical assistant. You must solve the problem step-by-step and be extremely concise, in English. "
  "Show your clear chain-of-thought reasoning. "
  "At the very end of your response, you MUST provide the final numerical answer inside explicit XML tags like this: <answer>YOUR_NUMERICAL_ANSWER</answer>. "
  "Do not include units or words inside the tag, only the absolute final number."
)

print("Formatting dataset...")
db1_distill_dataset = db1_distill_dataset.map(apply_chat_template)

sft_peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                             target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

sft_training_args = SFTConfig(output_dir="./qwen_sft_checkpoints", per_device_train_batch_size=8, gradient_accumulation_steps=2, learning_rate=1e-4, warmup_ratio=0.1,
                              lr_scheduler_type="cosine", num_train_epochs=5, logging_steps=10, save_strategy="epoch", bf16=True, report_to="none", max_length=1536,
                              dataset_text_field="text")

sft_trainer = SFTTrainer(model=sft_model, train_dataset=db1_distill_dataset, peft_config=sft_peft_config, processing_class=sft_tokenizer, args=sft_training_args)

print(f"Starting SFT training of {base_model_name}...")
sft_trainer.train()

sft_model_save_path = "/content/qwen-3.5-4b-sft-final"
sft_trainer.save_model(sft_model_save_path)
sft_tokenizer.save_pretrained(sft_model_save_path)

print(f"SFT training is completed, and the adapter weights are saved to {sft_model_save_path}.")

In [ ]:
#@title Run these before GRPO sessions (Proper Unsloth Installation)
## Install Unsloth and Update Libraries
"""
!pip install unsloth vllm
!pip install --upgrade pillow
!pip install --upgrade trl peft accelerate bitsandbytes
"""
## Update Transformers for Qwen-3.5-4B
"""
!pip install --upgrade transformers
"""
## Unsloth Correction after Colab Update
"""
!pip install --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade trl
"""
## TorchAO Correction after Colab Update
"""
!pip install --upgrade torch
"""

In [ ]:
#@title Standard GRPO Training (Model mA - Correctness)
import torch
import re
from datasets import load_dataset
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)
from trl import GRPOConfig, GRPOTrainer

def format_dataset(example):
  messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": example['question']}
  ]
  example["prompt"] = std_grpo_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  example["ground_truth"] = str(example["ground_truth"]).strip()

  return example

def extract_xml_answer(text):
  matches = re.findall(r'<answer>(.*?)</answer>', str(text), flags=re.DOTALL)
  if not matches:
    return None

  raw_answer = matches[-1].strip()
  number_match = re.search(r'[-+]?\d*\.?\d+', raw_answer.replace(',', ''))

  if number_match:
    clean_num = number_match.group(0)
    if clean_num.endswith('.0'):
      clean_num = clean_num[:-2]
    return clean_num

  return None

def format_reward_func(completions, **kwargs):
  rewards = []
  for completion in completions:
    text = completion[0]["content"] if isinstance(completion, list) else completion
    if "<answer>" in text and "</answer>" in text:
      rewards.append(1.0)
    else:
    rewards.append(0.0)

  return rewards

def accuracy_reward_func(completions, teacher_answer, **kwargs):
  rewards = []
  for content, truth in zip(completions, teacher_answer):
    text = content[0]["content"] if isinstance(content, list) else content

  student_answer = extract_xml_answer(text)
  teacher_answer = extract_xml_answer(truth)

  if str(student_answer) == str(teacher_answer) and student_answer is not None:
    rewards.append(2.0)
  elif teacher_answer is not None and str(teacher_answer) in text:
    rewards.append(1.0)
  else:
    rewards.append(0.0)

  return rewards

def brevity_reward_func(completions, **kwargs):
  rewards = []
  for completion in completions:
    text = completion[0]["content"] if isinstance(completion, list) else completion
    penalty = len(text) * 0.0005
    rewards.append(-penalty)

  return rewards

std_grpo_model_path = "/content/qwen-3.5-4b-sft-final"

print("Loading SFT Model with Unsloth...")
std_grpo_model, std_grpo_tokenizer = FastLanguageModel.from_pretrained(model_name=std_grpo_model_path, max_seq_length=4096, load_in_4bit=False,
                                                                       fast_inference=False, max_lora_rank=16)

print("Merging SFT adapters permanently into the base model...")
std_grpo_model = std_grpo_model.merge_and_unload()

std_grpo_model = FastLanguageModel.get_peft_model(std_grpo_model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                                                  lora_alpha=32, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth")

std_grpo_model.generation_config.max_new_tokens = 2048
std_grpo_model.generation_config.pad_token_id = std_grpo_tokenizer.eos_token_id

system_prompt = (
    "You are an expert mathematical assistant. You must solve the problem step-by-step and be extremely concise, in English. "
    "Show your clear chain-of-thought reasoning. "
    "At the very end of your response, you MUST provide the final numerical answer inside explicit XML tags like this: <answer>YOUR_NUMERICAL_ANSWER</answer>. "
    "Do not include units or words inside the tag, only the absolute final number."
)

print("Loading and formatting GRPO dataset...")
grpo_dataset = load_dataset("json", data_files="db2_grpo_cleaned_dataset.jsonl", split="train")
grpo_dataset = grpo_dataset.map(format_dataset)

std_grpo_training_args = GRPOConfig(output_dir="qwen_grpo_checkpoints", learning_rate=5e-6, lr_scheduler_type="cosine", per_device_train_batch_size=1,
                                    gradient_accumulation_steps=4, num_train_epochs=1, logging_steps=1, save_steps=50, bf16=True, num_generations=5, temperature=0.7,
                                    report_to="none", use_vllm=False)

std_grpo_trainer = GRPOTrainer(model=std_grpo_model, processing_class=std_grpo_tokenizer, reward_funcs=[format_reward_func, accuracy_reward_func, brevity_reward_func],
                               args=std_grpo_training_args, train_dataset=grpo_dataset, max_prompt_length=1024, max_completion_length=2048)

print("Starting Standard GRPO Training...")
std_grpo_trainer.train()

final_save_path = "/content/qwen-3.5-4b-std-grpo-final"
std_grpo_trainer.save_model(final_save_path)
std_grpo_tokenizer.save_pretrained(final_save_path)
print(f"GRPO Complete! Model saved to {final_save_path}")

In [ ]:
#@title Custom GRPO Training (Model mB - Correctness + Teacher Divergence)
import torch
import re
from datasets import load_dataset
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)
from trl import GRPOConfig, GRPOTrainer

def format_dataset(example):
  messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": example['question']}
  ]
  example["prompt"] = cstm_grpo_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  example["ground_truth"] = str(example["ground_truth"]).strip()

  return example

def extract_xml_answer(text):
  match = re.search(r"<answer>(.*?)</answer>", str(text), re.IGNORECASE | re.DOTALL)
  if match:
    val = match.group(1).strip().replace(",", "")
    if val.endswith('.0'): val = val[:-2]
    return val

  return None

def format_reward_func(completions, **kwargs):
  rewards = []
  for completion in completions:
    text = completion[0]["content"] if isinstance(completion, list) else completion
    if "<answer>" in text and "</answer>" in text:
      rewards.append(1.0)
    else:
      rewards.append(0.0)

  return rewards

def accuracy_reward_func(completions, teacher_answer, **kwargs):
  rewards = []
  for content, truth in zip(completions, teacher_answer):
    text = content[0]["content"] if isinstance(content, list) else content

    student_answer = extract_xml_answer(text)
    teacher_answer = extract_xml_answer(truth)

    if str(student_answer) == str(teacher_answer) and student_answer is not None:
      rewards.append(2.0)
    elif teacher_answer is not None and str(teacher_answer) in text:
      rewards.append(1.0)
    else:
      rewards.append(0.0)

  return rewards

def brevity_reward_func(completions, **kwargs):
  rewards = []
  for completion in completions:
    text = completion[0]["content"] if isinstance(completion, list) else completion
    penalty = len(text) * 0.0005
    rewards.append(-penalty)

  return rewards

def teacher_divergence_reward_func(completions, teacher_answer, **kwargs):
  rewards = []
  for content, truth in zip(completions, teacher_answer):
    student_text = content[0]["content"] if isinstance(content, list) else content
    teacher_text = str(truth)

    student_words = set(student_text.lower().split())
    teacher_words = set(teacher_text.lower().split())

    if not student_words or not teacher_words:
      rewards.append(0.0)
      continue

    intersection = student_words.intersection(teacher_words)
    union = student_words.union(teacher_words)
    similarity = len(intersection) / len(union)
    divergence = 1.0 - similarity

    if divergence > 0.85:
      rewards.append(1.5)
    elif divergence < 0.30:
      rewards.append(-1.0)
    else:
      rewards.append(0.5)

  return rewards

cstm_grpo_model_path = "/content/qwen-3.5-4b-sft-final"

print("Loading SFT Model with Unsloth...")
cstm_grpo_model, cstm_grpo_tokenizer = FastLanguageModel.from_pretrained(model_name=cstm_grpo_model_path, max_seq_length=4096, load_in_4bit=False,
                                                                         fast_inference=False, max_lora_rank=16)

print("Merging SFT adapters permanently into the base model...")
cstm_grpo_model = cstm_grpo_model.merge_and_unload()

cstm_grpo_model = FastLanguageModel.get_peft_model(cstm_grpo_model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                                                   lora_alpha=32, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth")

cstm_grpo_model.generation_config.max_new_tokens = 2048
cstm_grpo_model.generation_config.pad_token_id = cstm_grpo_tokenizer.eos_token_id

system_prompt = (
    "You are an expert mathematical assistant. You must solve the problem step-by-step and be extremely concise, in English. "
    "Show your clear chain-of-thought reasoning. "
    "At the very end of your response, you MUST provide the final numerical answer inside explicit XML tags like this: <answer>YOUR_NUMERICAL_ANSWER</answer>. "
    "Do not include units or words inside the tag, only the absolute final number."
)

print("Loading and formatting GRPO dataset...")
grpo_dataset = load_dataset("json", data_files="db2_grpo_cleaned_dataset.jsonl", split="train")
grpo_dataset = grpo_dataset.map(format_dataset)

cstm_grpo_training_args = GRPOConfig(output_dir="qwen_grpo_checkpoints", learning_rate=5e-6, lr_scheduler_type="cosine", per_device_train_batch_size=1,
                                     gradient_accumulation_steps=4, num_train_epochs=1, logging_steps=1, save_steps=50, bf16=True, num_generations=5, temperature=0.7,
                                     report_to="none", use_vllm=False, beta=0.04)

cstm_grpo_trainer = GRPOTrainer(model=cstm_grpo_model, processing_class=cstm_grpo_tokenizer, reward_funcs=[format_reward_func, accuracy_reward_func,
                                brevity_reward_func, teacher_divergence_reward_func], args=cstm_grpo_training_args, train_dataset=grpo_dataset,
                                max_prompt_length=1024, max_completion_length=2048)

print("Starting Custom GRPO Training...")
cstm_grpo_trainer.train()

final_save_path = "/content/qwen-3.5-4b-cstm-grpo-final"
cstm_grpo_trainer.save_model(final_save_path)
cstm_grpo_tokenizer.save_pretrained(final_save_path)
print(f"GRPO Complete! Model saved to {final_save_path}")

In [ ]:
#@title Connect to Google Drive to Pull Models
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Model Evaluation and Graph Generation
import torch
import gc
import re
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
from datasets import load_dataset, load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

def extract_model_answer_xml(text):
  match = re.search(r"<answer>(.*?)</answer>", str(text), re.IGNORECASE | re.DOTALL)
  if match:
    value = match.group(1).strip().replace(",", "")
    if value.endswith('.0'): value = value[:-2]
    return value
  return None

def extract_dataset_answer_regex(truth_text):
  matches = re.findall(r'[-+]?\d+(?:[.,\s]\d+)*', str(truth_text))
  if not matches:
    return None
  raw_number = matches[-1].strip()
  clean_number = raw_number.replace(' ', '')

  if '.' in clean_number and ',' not in clean_number:
    parts = clean_number.split('.')
    if all(len(p) == 3 for p in parts[1:]): clean_number = clean_number.replace('.', '')
  if ',' in clean_number and '.' not in clean_number:
    parts = clean_number.split(',')
    if all(len(p) == 3 for p in parts[1:]): clean_number = clean_number.replace(',', '')
  if clean_number.endswith('.0'): clean_number = clean_number[:-2]
  if clean_number.endswith(',0'): clean_number = clean_number[:-2]
  return clean_number

def compare_math_answers(model_text, teacher_text):
  model_answer = extract_model_answer_xml(model_text)
  teacher_answer = extract_dataset_answer_regex(teacher_text)

  if model_answer is None or teacher_answer is None:
    return False
  return str(model_answer) == str(teacher_answer)

print("Loading test models...")
qwen_sft_model_path = "/content/drive/MyDrive/ytu_ce_comp_sem_models/qwen-3.5-4b-sft-final"
qwen_std_grpo_model_path = "/content/drive/MyDrive/ytu_ce_comp_sem_models/qwen-3.5-4b-std-grpo-final"
qwen_cstm_grpo_model_path = "/content/drive/MyDrive/ytu_ce_comp_sem_models/qwen-3.5-4b-cstm-grpo-final"

print("Loading db3 dataset...")
db3_path = "/content/data/db3_test"

try:
  loaded_dataset = load_from_disk(db3_path)
  print("Dataset loaded from disk.")
except:
  loaded_dataset = load_dataset("ytu-ce-cosmos/gsm8k_tr", split="train")
  loaded_dataset = loaded_dataset.shuffle(seed=59)
  loaded_dataset = loaded_dataset.select(range(1100, 1600))
  print("Dataset loaded from Huggingface.")

db3 = {
  "questions": loaded_dataset["question"],
  "answers": loaded_dataset["answer"]
}

print(f"Successfully loaded the test models and {len(db3['questions'])} questions from db3.")

system_prompt = (
  "You are an expert mathematical assistant. You must solve the problem step-by-step and be extremely concise, in English. "
  "Show your clear chain-of-thought reasoning. "
  "At the very end of your response, you MUST provide the final numerical answer inside explicit XML tags like this: <answer>YOUR_NUMERICAL_ANSWER</answer>. "
  "Do not include units or words inside the tag, only the absolute final number."
)

evaluation_target_models = [
  ("SFT Trained Model", qwen_sft_model_path),
  ("Standard GRPO Trained Model", qwen_std_grpo_model_path),
  ("Custom GRPO Trained Model", qwen_cstm_grpo_model_path)
]
generated_answers = {}
batch_size = 32
variant_count = 5

for model_name, model_path in evaluation_target_models:
  print(f"Initializing {model_name} via Transformers...")

  tokenizer = AutoTokenizer.from_pretrained(model_path)
  tokenizer.padding_side = "left"
  if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

  model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map="auto")
  model.eval()

  model_generated_blocks = []
  questions = db3["questions"]

  print(f"Starting batch inference procedure with batch size of {batch_size} and variant count of {variant_count}...")

  for i in tqdm(range(0, len(questions), batch_size)):
    batch_questions = questions[i:i+batch_size]

    batch_prompts = []
    for question in batch_questions:
      messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
      ]
      prompt_string = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
      batch_prompts.append(prompt_string)

    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
      outputs = model.generate(**inputs, max_new_tokens=1536, use_cache=True, temperature=0.7, do_sample=True, num_return_sequences=variant_count,
                               pad_token_id=tokenizer.eos_token_id)

    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[:, input_length:]

    flat_decoded_answers = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

    for j in range(0, len(flat_decoded_answers), variant_count):
      question_variants = flat_decoded_answers[j:j + variant_count]
      model_generated_blocks.append(question_variants)

  generated_answers[model_name] = model_generated_blocks

  del model
  del tokenizer
  torch.cuda.empty_cache()

print("Loading Qwen3-Embedding-0.6B model...")
embedding_model = "Qwen/Qwen3-Embedding-0.6B"
text_embedder = SentenceTransformer(embedding_model, trust_remote_code=True)

final_eval_scores = {}
test_failures = []

print("Starting the accuracy and diversity evaluations of all models...")
for model_name, all_question_answers in generated_answers.items():
  print(f"Currently evaluating: {model_name}")
  model_accuracies = []
  model_diversities = []

  for question_index, all_answer_variants in enumerate(all_question_answers):
    dataset_answer = db3["answers"][question_index]

    correct_count = 0
    for answer in all_answer_variants:
      if compare_math_answers(answer, dataset_answer):
        correct_count += 1
      elif model_name == "Custom GRPO Trained Model" and len(test_failures) < 50:
        test_failures.append({"truth": dataset_answer, "model": answer})

    model_accuracies.append(correct_count / len(all_answer_variants))

    formatted_answers = [str(ans) for ans in all_answer_variants]
    answer_embeddings = text_embedder.encode(formatted_answers)

    similarity_scores = []
    for i, j in combinations(range(len(all_answer_variants)), 2):
      similarity = cosine_similarity(answer_embeddings[i].reshape(1, -1), answer_embeddings[j].reshape(1, -1))[0][0]
      similarity_scores.append(similarity)

    average_similarity = np.mean(similarity_scores) if similarity_scores else 1.0
    model_diversities.append(1.0 - average_similarity)

  final_eval_scores[model_name] = {
    "Accuracies": model_accuracies,
    "Diversities": model_diversities,
    "Mean Acc. of Model": np.mean(model_accuracies),
    "Mean Div. of Model": np.mean(model_diversities)
  }

print("=====DB3 FINAL EVALUATION REPORT=====")
for model_name, scores in final_eval_scores.items():
  print(f"{model_name} Results:")
  print(f"Mean Accuracy : {scores['Mean Acc. of Model']*100:.2f}%")
  print(f"Mean Diversity: {scores['Mean Div. of Model']:.4f}")

if test_failures:
  # Extracting failure examples from the custom GRPO model for the presentation and report.
  print("-"*75)
  print("Reward Hacking Snippet for the Custom GRPO Model")
  print(f"Expected Target: {extract_dataset_answer_regex(test_failures[0]['truth'])}")
  print(f"Actual Output: {test_failures[0]['model'][-250:]}")

### NOTE: LLM Assistance was used to change parts of the code onwards to provide a more readable graph.
print("Generating thesis data plots...")
num_of_models = len(final_eval_scores)
if num_of_models == 0:
  print("No model evaluations processed. Ending execution.")
else:
  fig, axes = plt.subplots(1, num_of_models, figsize=(6 * num_of_models, 5), squeeze=False)
  axes = axes.flatten()

  colors = ['#ff9999', '#66b3ff', '#99ff99']
  edge_colors = ['darkred', 'darkblue', 'darkgreen']

  for index, (model_name, scores) in enumerate(final_eval_scores.items()):
    ax = axes[index]
    x = scores["Diversities"]
    y = scores["Accuracies"]
    mean_X = scores["Mean Div. of Model"]
    mean_Y = scores["Mean Acc. of Model"]

    ax.scatter(x, y, color=colors[index % len(colors)], alpha=0.4, edgecolors='none', s=60, label="db3 Questions")
    ax.scatter(mean_X, mean_Y, color=edge_colors[index % len(edge_colors)], s=250, label="Model Center Point", edgecolors='black', linewidth=1.5, zorder=5)
    ax.scatter(1.0, 1.0, color='gold', marker='*', s=400, label="Theoretical Peak", edgecolors='black', zorder=6)

    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"{model_name}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Reasoning Diversity (1 - Cosine Similarity)", fontsize=11)
    if index == 0:
      ax.set_ylabel("Mathematical Accuracy", fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='lower right', fontsize=9)

  plt.suptitle("Evaluation of All Models Using db3 Questions: Reasoning Diversity vs. Mathematical Accuracy", fontsize=15, fontweight='bold', y=1.05)
  plt.tight_layout()

  output_image = "db3_model_comparison_plots.png"
  plt.savefig(output_image, dpi=300, bbox_inches='tight')
  plt.show()
  print(f"Evaluation process successful. Final performance plots exported as '{output_image}'.")